## Assignment: GenAI – AI Resume Screening System with Tracing

**Pipeline:**  
`Resume -> Skill Extraction -> Matching -> Scoring -> Explanation -> Tracing`

**Tools Used:** Python . LangChain . LangSmith . grok

## Step 0: Install Dependencies

In [1]:
pip install langchain langchain-groq langchain-core langsmith python-dotenv --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/513.3 kB ? eta -:--:--
    --------------------------------------- 10.2/513.3 kB ? eta -:--:--
    --------------------------------------- 10.2/513.3 kB ? eta -:--:--
   -- ------------------------------------ 30.7/513.3 kB 163.8 kB/s eta 0:00:03
   -- ------------------------------------ 30.7/513.3 kB 163.8 kB/s eta 0:00:03
   ---- ---------------------------------- 61.4/51


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Setup — API Keys & LangSmith Tracing

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)   

# ─ LangSmith Tracing Configuration ─
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "your_langsmith_api_key")
os.environ["LANGCHAIN_PROJECT"] = "AI-Resume-Screening-System"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# OpenAI Key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your_openai_api_key")

print(" Environment configured.")
print(f"LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"Tracing Enabled: {os.environ['LANGCHAIN_TRACING_V2']}")

 Environment configured.
LangSmith Project: AI-Resume-Screening-System
Tracing Enabled: true


## Step 2: Import Libraries & Initialize LLM

In [3]:
# NEW
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableLambda
from langsmith import traceable

# Initializing LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY")
)
print("LLM initialized: groq llama3-8b-8192")

C:\Users\Zaara\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM initialized: groq llama3-8b-8192


## Step 3: Define Prompt Templates

In [4]:
# Prompt 1: Skill Extraction
# Extracts skills, tools, experience, education from raw resume text

SKILL_EXTRACTION_TEMPLATE = """
You are a precise resume parser. Your job is to extract structured information 
from the resume text below.

IMPORTANT RULES:
- Only extract skills, tools, and experience that are EXPLICITLY stated in the resume.
- Do NOT infer or assume anything not written in the resume.
- If a field has no data, return "Not mentioned".

Resume Text:
{resume_text}

Extract and return the following in this exact format:

SKILLS: [comma-separated list of technical skills]
TOOLS: [comma-separated list of tools/technologies/frameworks]
EXPERIENCE: [brief summary of total experience and roles held]
EDUCATION: [highest qualification and institution if mentioned]
PROJECTS: [comma-separated list of notable project names or "Not mentioned"]
CERTIFICATIONS: [comma-separated list or "Not mentioned"]
"""

skill_extraction_prompt = PromptTemplate(
    input_variables=["resume_text"],
    template=SKILL_EXTRACTION_TEMPLATE
)

print(" Skill extraction prompt defined.")

 Skill extraction prompt defined.


In [5]:
# ── Prompt 2: Matching + Scoring + Explanation ──────────────────────────
# This is the core evaluation prompt.
# Uses few-shot scoring guide to help the model understand score ranges (bonus feature)

MATCHING_SCORING_TEMPLATE = """
You are an expert technical recruiter evaluating candidate fit for a job role.

Job Description:
{job_description}

Candidate Profile (extracted from resume):
{extracted_profile}

Your task:
1. Compare the candidate's skills, tools, and experience with the job requirements.
2. Identify which required skills are PRESENT and which are MISSING.
3. Assign a FIT SCORE between 0 and 100 based on how well the candidate matches.
4. Provide a clear, honest explanation for the score.

Scoring Guide (Few-Shot Examples):
- A candidate with Python, ML, TensorFlow, SQL, 3 years experience → Score ~88
- A candidate with Python, basic ML, SQL, 1 year internship → Score ~55
- A candidate with only Excel and HTML, no ML experience → Score ~15

IMPORTANT RULES:
- Base your score ONLY on information in the candidate profile above.
- Do NOT assume the candidate has skills not listed in their profile.
- Be honest and direct.

Return your response in this EXACT format:

MATCHED_SKILLS: [skills present in both resume and job description]
MISSING_SKILLS: [skills required by job but absent in resume]
FIT_SCORE: [integer between 0-100]
SCORE_EXPLANATION: [2-4 sentences explaining why this score was given, referencing specific skills and gaps]
HIRING_RECOMMENDATION: [one of: SHORTLIST / CONSIDER / REJECT]
"""

matching_scoring_prompt = PromptTemplate(
    input_variables=["job_description", "extracted_profile"],
    template=MATCHING_SCORING_TEMPLATE
)

print(" Matching + scoring prompt defined.")

 Matching + scoring prompt defined.


## Step 4: Build LCEL Chains

In [6]:
# LCEL Chain 1: Skill Extraction
# Flow: PromptTemplate → LLM
extraction_chain = skill_extraction_prompt | llm

# LCEL Chain 2: Matching + Scoring + Explanation
# Flow: PromptTemplate → LLM
matching_chain = matching_scoring_prompt | llm

print(" Extraction chain built.")
print(" Matching + scoring chain built.")
print("\nBoth chains use LCEL pipe (|) syntax as per LangChain documentation.")

 Extraction chain built.
 Matching + scoring chain built.

Both chains use LCEL pipe (|) syntax as per LangChain documentation.


## Step 5: Define Input Data

### Job Description + 3 Resumes (Strong / Average / Weak)

In [7]:
# -- Job Description --
JOB_DESCRIPTION = """
Position: Data Scientist
Company: TechCorp Analytics

Required Skills:
- Python (pandas, numpy, scikit-learn)
- Machine Learning (supervised and unsupervised)
- Deep Learning (TensorFlow or PyTorch)
- SQL and database querying
- Data visualization (matplotlib, seaborn, or Tableau)
- Feature engineering and model evaluation

Nice to Have:
- Experience with cloud platforms (AWS / GCP / Azure)
- NLP or computer vision experience
- MLOps / model deployment experience

Experience Required: 2+ years in a data science or ML role
Education: Bachelor's or Master's in Computer Science, Statistics, or related field
"""

# -- Resume 1: Strong Candidate--
RESUME_STRONG = """
Name: Ananya Sharma
Email: ananya.sharma@email.com

Summary:
Data Scientist with 3 years of experience building ML models and deploying them at scale.

Skills:
Python, pandas, numpy, scikit-learn, TensorFlow, PyTorch, SQL, matplotlib, seaborn, Tableau,
feature engineering, model evaluation, A/B testing, NLP, AWS SageMaker, MLflow

Experience:
- Data Scientist at Flipkart Analytics (2022–Present, 2 years)
  Built recommendation engine using collaborative filtering, improved CTR by 18%.
  Deployed ML models on AWS SageMaker with MLflow tracking.
  
- Junior Data Scientist at InfoEdge (2021–2022, 1 year)
  Worked on churn prediction using Random Forest and XGBoost.
  Wrote SQL queries for large-scale data extraction.

Education:
M.Tech in Data Science — IIT Bombay, 2021

Certifications:
IBM Data Science Professional Certificate, AWS Certified Machine Learning Specialty

Projects:
- NLP Sentiment Analysis Pipeline (BERT-based)
- Customer Churn Prediction Dashboard
- Image Classification using CNN (PyTorch)
"""

# -- Resume 2: Average Candidate ---
RESUME_AVERAGE = """
Name: Ravi Mehta
Email: ravi.mehta@email.com

Summary:
Aspiring data scientist with 1 year of internship experience and strong academic background.

Skills:
Python, pandas, numpy, scikit-learn, SQL, matplotlib

Experience:
- Data Science Intern at Infosys (2023–2024, 10 months)
  Worked on regression models for sales forecasting.
  Used Python and pandas for data cleaning and EDA.
  Created visualizations using matplotlib.

Education:
B.E. in Computer Science — Pune University, 2023

Projects:
- House Price Prediction using Linear Regression
- COVID-19 Data Analysis (Exploratory)
"""

# -- Resume 3: Weak Candidate --
RESUME_WEAK = """
Name: Karan Patil
Email: karan.patil@email.com

Summary:
Fresh graduate looking for entry level opportunities in IT.

Skills:
MS Excel, PowerPoint, basic Python (loops, functions), HTML, CSS

Experience:
No professional work experience.

Education:
BCA — Nagpur University, 2024

Projects:
- College website using HTML and CSS
- Excel-based attendance tracker
"""

print(" Job description and all 3 resumes loaded.")

 Job description and all 3 resumes loaded.


## Step 6: Run the Full Pipeline

In [8]:
@traceable(name="resume_screening_pipeline")
def run_screening_pipeline(candidate_name, resume_text, job_description):
    """
    Full 4-step screening pipeline for a single resume.
    
    Step 1: Extract skills/experience from resume text
    Step 2: Match against job description requirements
    Step 3: Assign a fit score 0-100
    Step 4: Generate explanation for the score
    """
    print(f"\n{'='*55}")
    print(f"  Candidate: {candidate_name}")
    print(f"{'='*55}")
    
    # -- Step 1: Skill Extraction (invoke the chain)--
    print("\n [Step 1] Extracting skills from resume...")
    extraction_result = extraction_chain.invoke({"resume_text": resume_text})
    extracted_profile = extraction_result.content
    print(extracted_profile)
    
    # -- Steps 2+3+4: Match → Score → Explain ---
    print("\n [Steps 2/3/4] Matching, scoring, and explaining...")
    scoring_result = matching_chain.invoke({
        "job_description": job_description,
        "extracted_profile": extracted_profile
    })
    score_output = scoring_result.content
    print(score_output)
    
    # -- Parse numeric score ---
    fit_score = None
    for line in score_output.split("\n"):
        if line.strip().startswith("FIT_SCORE:"):
            try:
                fit_score = int(line.replace("FIT_SCORE:", "").strip())
            except ValueError:
                fit_score = 0
            break
    
    return {
        "candidate": candidate_name,
        "extracted_profile": extracted_profile,
        "score_result": score_output,
        "fit_score": fit_score
    }

print(" Pipeline function defined with @traceable decorator.")

 Pipeline function defined with @traceable decorator.


In [9]:
# - Run 1: Strong Candidate ---
result_strong = run_screening_pipeline(
    candidate_name="Strong Candidate — Ananya Sharma",
    resume_text=RESUME_STRONG,
    job_description=JOB_DESCRIPTION
)


  Candidate: Strong Candidate — Ananya Sharma

 [Step 1] Extracting skills from resume...
SKILLS: feature engineering, model evaluation, A/B testing, NLP 
TOOLS: Python, pandas, numpy, scikit-learn, TensorFlow, PyTorch, SQL, matplotlib, seaborn, Tableau, AWS SageMaker, MLflow 
EXPERIENCE: 3 years of experience as a Data Scientist at Flipkart Analytics and Junior Data Scientist at InfoEdge 
EDUCATION: M.Tech in Data Science — IIT Bombay 
PROJECTS: NLP Sentiment Analysis Pipeline, Customer Churn Prediction Dashboard, Image Classification using CNN 
CERTIFICATIONS: IBM Data Science Professional Certificate, AWS Certified Machine Learning Specialty

 [Steps 2/3/4] Matching, scoring, and explaining...
MATCHED_SKILLS: Python, pandas, numpy, scikit-learn, Machine Learning, Deep Learning (TensorFlow or PyTorch), SQL, data visualization (matplotlib, seaborn, Tableau), feature engineering, model evaluation, NLP
MISSING_SKILLS: None explicitly mentioned, but experience with cloud platforms (AWS 

In [10]:
# - Run 2: Average Candidate ---
result_average = run_screening_pipeline(
    candidate_name="Average Candidate — Ravi Mehta",
    resume_text=RESUME_AVERAGE,
    job_description=JOB_DESCRIPTION
)


  Candidate: Average Candidate — Ravi Mehta

 [Step 1] Extracting skills from resume...
SKILLS: Data Science, Regression Models, Data Cleaning, EDA, Data Analysis, Sales Forecasting, 
TOOLS: Python, pandas, numpy, scikit-learn, SQL, matplotlib, 
EXPERIENCE: 1 year of internship experience as a Data Science Intern, 
EDUCATION: B.E. in Computer Science — Pune University, 
PROJECTS: House Price Prediction using Linear Regression, COVID-19 Data Analysis, 
CERTIFICATIONS: Not mentioned

 [Steps 2/3/4] Matching, scoring, and explaining...
MATCHED_SKILLS: Python, pandas, numpy, scikit-learn, SQL, matplotlib, Machine Learning (basic), Data Analysis
MISSING_SKILLS: Deep Learning (TensorFlow or PyTorch), Data visualization (seaborn or Tableau), Feature engineering, Model evaluation, Cloud platforms, NLP, Computer Vision, MLOps
FIT_SCORE: 55
SCORE_EXPLANATION: The candidate has a solid foundation in Python, Machine Learning, and data analysis, which aligns with some of the job requirements. Howe

In [11]:
# - Run 3: Weak Candidate ---
result_weak = run_screening_pipeline(
    candidate_name="Weak Candidate — Karan Patil",
    resume_text=RESUME_WEAK,
    job_description=JOB_DESCRIPTION
)


  Candidate: Weak Candidate — Karan Patil

 [Step 1] Extracting skills from resume...
SKILLS: MS Excel, PowerPoint, Python, HTML, CSS
TOOLS: Not mentioned
EXPERIENCE: No professional work experience
EDUCATION: BCA — Nagpur University
PROJECTS: College website, Excel-based attendance tracker
CERTIFICATIONS: Not mentioned

 [Steps 2/3/4] Matching, scoring, and explaining...
MATCHED_SKILLS: Python
MISSING_SKILLS: Machine Learning, Deep Learning, SQL, data visualization, feature engineering, model evaluation, and relevant experience
FIT_SCORE: 10
SCORE_EXPLANATION: The candidate's profile shows only a basic skill set with Python, which is a required skill, but lacks all other essential skills for the Data Scientist role, including Machine Learning, Deep Learning, and SQL. The candidate also has no professional experience, which is a significant requirement for the position. The presence of Python is the only redeeming factor, but it is not enough to compensate for the numerous gaps in the

## Step 7: Final Summary Table

In [12]:
import pandas as pd

# Parse recommendation from score result
def get_recommendation(score_result):
    for line in score_result.split("\n"):
        if "HIRING_RECOMMENDATION" in line:
            return line.split(":")[-1].strip()
    return "N/A"

summary_data = [
    {
        "Candidate": r["candidate"],
        "Fit Score": r["fit_score"],
        "Recommendation": get_recommendation(r["score_result"])
    }
    for r in [result_strong, result_average, result_weak]
]

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("FINAL SCREENING SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))


FINAL SCREENING SUMMARY
                       Candidate  Fit Score Recommendation
Strong Candidate — Ananya Sharma         95      SHORTLIST
  Average Candidate — Ravi Mehta         55       CONSIDER
    Weak Candidate — Karan Patil         10         REJECT


## Step 8: Deliberate Incorrect Output (LangSmith Debug)

In [13]:
# - Intentional Bad Run: Debugging Example ---
# This run is intentionally given a vague, incomplete resume to show
# what a bad/incorrect output looks like in LangSmith tracing.
# LangSmith would flag this run and we can inspect the prompt + output.

RESUME_BAD_INPUT = """
Name: Unknown Person
I am good at stuff and work hard. I know computers.
"""

print(" Running intentionally bad input to demonstrate LangSmith debugging...")
print("Expected: LLM will struggle to extract meaningful skills → low quality output")
print("This trace will be visible in LangSmith for debugging.\n")

result_bad = run_screening_pipeline(
    candidate_name="DEBUG — Incomplete Resume (Intentional Bad Input)",
    resume_text=RESUME_BAD_INPUT,
    job_description=JOB_DESCRIPTION
)

print("\n Note: This run demonstrates why input validation is important.")
print(" In LangSmith, you can see this trace shows 'Not mentioned' for most fields.")
print(" Fix: Add input validation before calling the chain to catch empty/vague resumes.")

 Running intentionally bad input to demonstrate LangSmith debugging...
Expected: LLM will struggle to extract meaningful skills → low quality output
This trace will be visible in LangSmith for debugging.


  Candidate: DEBUG — Incomplete Resume (Intentional Bad Input)

 [Step 1] Extracting skills from resume...
SKILLS: computers
TOOLS: Not mentioned
EXPERIENCE: Not mentioned
EDUCATION: Not mentioned
PROJECTS: Not mentioned
CERTIFICATIONS: Not mentioned

 [Steps 2/3/4] Matching, scoring, and explaining...
MATCHED_SKILLS: None
MISSING_SKILLS: Python, Machine Learning, Deep Learning, SQL, data visualization, feature engineering, model evaluation, and all other required skills
FIT_SCORE: 0
SCORE_EXPLANATION: The candidate's profile does not mention any relevant skills, tools, or experience that match the job requirements. The lack of information about their technical abilities, education, and work experience makes it impossible to assess their fit for the Data Scientist role. As a result, 

## Step 9: Structured JSON Output (Bonus Feature)

In [14]:
import json

def parse_to_json(result):
    """Parse the raw score output into a structured JSON object."""
    parsed = {
        "candidate": result["candidate"],
        "fit_score": result["fit_score"],
        "recommendation": get_recommendation(result["score_result"]),
        "raw_score_output": result["score_result"]
    }
    
    # Extract matched and missing skills
    for line in result["score_result"].split("\n"):
        if line.startswith("MATCHED_SKILLS:"):
            parsed["matched_skills"] = line.replace("MATCHED_SKILLS:", "").strip()
        elif line.startswith("MISSING_SKILLS:"):
            parsed["missing_skills"] = line.replace("MISSING_SKILLS:", "").strip()
        elif line.startswith("SCORE_EXPLANATION:"):
            parsed["explanation"] = line.replace("SCORE_EXPLANATION:", "").strip()
    
    return parsed

# Generate JSON outputs for all results
json_results = [parse_to_json(r) for r in [result_strong, result_average, result_weak]]

print("Structured JSON Output (Bonus Feature):")
print(json.dumps(json_results, indent=2))

Structured JSON Output (Bonus Feature):
[
  {
    "candidate": "Strong Candidate \u2014 Ananya Sharma",
    "fit_score": 95,
    "recommendation": "SHORTLIST",
    "raw_score_output": "MATCHED_SKILLS: Python, pandas, numpy, scikit-learn, Machine Learning, Deep Learning (TensorFlow or PyTorch), SQL, data visualization (matplotlib, seaborn, Tableau), feature engineering, model evaluation, NLP\nMISSING_SKILLS: None explicitly mentioned, but experience with cloud platforms (AWS / GCP / Azure) is only partially covered by AWS SageMaker\nFIT_SCORE: 95\nSCORE_EXPLANATION: The candidate has an exceptional match with the job requirements, possessing all the required skills including Python, Machine Learning, Deep Learning, SQL, and data visualization. The candidate's experience with NLP and certifications like AWS Certified Machine Learning Specialty are significant advantages. The only minor gap is the limited mention of cloud platform experience, but AWS SageMaker and MLflow experience partia